# 🟠 Stage 3 Instruction Training (Ubuntu/Linux)

Linux-first notebook for instruction tuning with the shared `instruction.16gb.yaml` config and `libs/instruction` trainer.

Run flow: 🧭 setup → ⚙️ config → 📁 data folders → 🔎 audit → 🚀 train → ✅ result.

In [ ]:
import json
import os
import platform
import sys
from pathlib import Path

import torch

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

ICONS = {
    "setup": "🧭",
    "linux": "🟠",
    "gpu": "⚡",
    "config": "⚙️",
    "data": "📁",
    "audit": "🔎",
    "train": "🚀",
    "done": "✅",
    "warn": "⚠️",
}


def linux_distribution_name() -> str:
    try:
        release = platform.freedesktop_os_release()
    except (AttributeError, OSError):
        return platform.system()
    return release.get("PRETTY_NAME") or release.get("NAME") or platform.system()


def find_repo_dir(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path
    repo_dir_env = os.environ.get("MDNAC_REPO_DIR")
    if repo_dir_env:
        repo_dir = Path(repo_dir_env).expanduser().resolve()
        if (repo_dir / "pyproject.toml").exists() and (repo_dir / "libs").is_dir():
            return repo_dir
    raise RuntimeError("Run this notebook from inside the repo or set MDNAC_REPO_DIR.")


def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line.startswith("export "):
            line = line[len("export "):].strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


REPO_DIR = find_repo_dir(Path.cwd())
os.chdir(REPO_DIR)
load_env_file(REPO_DIR / ".env")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.core.pretrain.training_config import _load_yaml_mapping
from libs.instruction import (
    InstructionTrainer,
    audit_instruction_jsonl,
    build_instruction_training_config,
)

print(f"{ICONS['setup']} repo = {REPO_DIR}")
print(f"{ICONS['linux']} os = {linux_distribution_name()} ({platform.system()} {platform.release()})")
print(f"{ICONS['gpu']} cuda_available = {torch.cuda.is_available()} | cuda_device_count = {torch.cuda.device_count()}")

In [ ]:
CONFIG_PATH = Path(os.environ.get("MDNAC_INSTRUCTION_CONFIG", REPO_DIR / "instruction.16gb.yaml")).expanduser()
if not CONFIG_PATH.is_absolute():
    CONFIG_PATH = (REPO_DIR / CONFIG_PATH).resolve()

raw_config = _load_yaml_mapping(CONFIG_PATH)
CONFIG = build_instruction_training_config(REPO_DIR, raw_config)
if platform.system() != "Linux":
    print(f"{ICONS['warn']} This notebook is Ubuntu/Linux-first. Use 01_instruction_train.ipynb for Windows/Jupyter adjustments.")
else:
    print(f"{ICONS['linux']} Ubuntu/Linux runtime detected")

config_preview = {
    "config_path": str(CONFIG_PATH),
    "instruction_jsonl_paths": [str(path) for path in CONFIG.instruction_jsonl],
    "artifact_source_jsonl": str(CONFIG.artifact_source_jsonl),
    "base_checkpoint_path": str(CONFIG.base_checkpoint_path),
    "output_dir": str(CONFIG.output_dir),
    "artifact_dir": str(CONFIG.artifact_dir),
    "optimizer": {
        "type": CONFIG.optimizer_type,
        "learning_rate": CONFIG.learning_rate,
        "muon_learning_rate": CONFIG.muon_learning_rate,
        "weight_decay": CONFIG.weight_decay,
        "fused": CONFIG.fused,
    },
    "runtime": {
        "platform": platform.system(),
        "cuda_device_count": torch.cuda.device_count(),
        "device": CONFIG.device,
        "multi_gpu_mode": CONFIG.multi_gpu_mode,
        "mixed_precision": CONFIG.mixed_precision,
        "world_size_env": os.environ.get("WORLD_SIZE"),
        "rank_env": os.environ.get("RANK"),
        "local_rank_env": os.environ.get("LOCAL_RANK"),
    },
    "data": {
        "batch_size": CONFIG.batch_size,
        "gradient_accumulation_steps": CONFIG.gradient_accumulation_steps,
        "num_workers": CONFIG.num_workers,
        "shuffle_buffer_size": CONFIG.shuffle_buffer_size,
    },
    "artifacts": {
        "profile_vocab_size": CONFIG.profile_vocab_size,
        "profile_sample_size": CONFIG.artifact_profile_sample_size,
        "sequence_tokenizer_map_path": str(CONFIG.sequence_tokenizer_map_path),
    },
}
print(f"{ICONS['config']} instruction training config")
print(json.dumps(config_preview, indent=2, ensure_ascii=False))

In [ ]:
def project_path(value) -> Path:
    path = Path(str(value)).expanduser()
    return path if path.is_absolute() else (REPO_DIR / path).resolve()


paths_config = raw_config.get("paths", {})
data_config = raw_config.get("data", {})
DATA_DIR = REPO_DIR / "data"
INSTRUCTION_JSONL = project_path(paths_config.get("instruction_jsonl_path", "data/compiled/refseq_bacteria_protein/instruction.jsonl"))
INSTRUCTION_PART_DIR = project_path(paths_config.get("instruction_part_dir", INSTRUCTION_JSONL.parent))
INSTRUCTION_PART_GLOB = str(data_config.get("instruction_part_glob") or "instruction_part_*.jsonl")
BASE_CHECKPOINT = Path(CONFIG.base_checkpoint_path)
OUTPUT_DIR = Path(CONFIG.output_dir)
ARTIFACT_DIR = Path(CONFIG.artifact_dir)
ARTIFACT_SOURCE_JSONL = Path(CONFIG.artifact_source_jsonl)

for folder in (DATA_DIR, INSTRUCTION_JSONL.parent, INSTRUCTION_PART_DIR, OUTPUT_DIR, ARTIFACT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

instruction_part_paths = sorted(INSTRUCTION_PART_DIR.glob(INSTRUCTION_PART_GLOB))
data_setup = {
    "data_dir": str(DATA_DIR),
    "compiled_instruction_dir": str(INSTRUCTION_JSONL.parent),
    "instruction_jsonl": {"path": str(INSTRUCTION_JSONL), "exists": INSTRUCTION_JSONL.exists()},
    "instruction_parts": {
        "dir": str(INSTRUCTION_PART_DIR),
        "glob": INSTRUCTION_PART_GLOB,
        "count": len(instruction_part_paths),
        "preview": [str(path) for path in instruction_part_paths[:5]],
    },
    "artifact_source_jsonl": {"path": str(ARTIFACT_SOURCE_JSONL), "exists": ARTIFACT_SOURCE_JSONL.exists()},
    "base_checkpoint": {"path": str(BASE_CHECKPOINT), "exists": BASE_CHECKPOINT.exists()},
    "output_dir": str(OUTPUT_DIR),
    "artifact_dir": str(ARTIFACT_DIR),
}
print(f"{ICONS['data']} data folder setup")
print(json.dumps(data_setup, indent=2, ensure_ascii=False))

In [ ]:
print(f"{ICONS['audit']} auditing instruction JSONL")
audit = audit_instruction_jsonl(
    CONFIG.instruction_jsonl,
    default_sequence_type=CONFIG.default_sequence_type,
    instruction_field=CONFIG.instruction_field,
    input_field=CONFIG.input_field,
    output_field=CONFIG.output_field,
)
print(json.dumps(audit.to_dict(), indent=2, ensure_ascii=False))

In [ ]:
print(f"{ICONS['train']} starting instruction training")
trainer = InstructionTrainer(CONFIG)
result = trainer.train()
print(f"{ICONS['done']} instruction training finished")
print(json.dumps(result.to_dict(), indent=2, ensure_ascii=False))